# GNN-Conditioned QLSTM — Training với lightning.gpu

**Config:**
- QLSTM backend: `lightning.gpu` + `adjoint` + `float64`
- QAOA backend: `lightning.qubit` + `adjoint` (tránh CUDA handle limit)
- QAOA depth: `p=2` | GNN: `dim_h=16`
- Checkpoint tự động lưu vào **Google Drive** sau mỗi epoch
- Auto-resume khi session bị ngắt

**Thứ tự chạy:**
1. Cell 1 — verify GPU
2. Cell 2 — install → **Restart runtime**
3. Cell 3 — upload zip
4. Cell 4 — extract
5. Cell 4b — mount Google Drive
6. Cell 5 — setup path + verify backend
7. Cell 6 — model config
8. Cell 7 — shape test
9. Cell 8 — training (tự resume nếu Drive có checkpoint)
10. Cell 9 — download checkpoint

In [ ]:
# Cell 1 — Verify GPU
import torch
assert torch.cuda.is_available(), 'GPU not available! Change runtime type to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('PyTorch:', torch.__version__)

In [ ]:
# Cell 2 — Install (sau khi xong: Runtime → Restart runtime)
!pip install -q pennylane pennylane-lightning-gpu cuquantum-python-cu12
!pip install -q torch_geometric
!pip install -q networkx matplotlib

import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '')
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

print('Done! >>> Restart runtime now.')

In [ ]:
# Cell 3 — Upload zip
from google.colab import files
uploaded = files.upload()   # chọn QLSTM_with_GNN.zip

In [ ]:
# Cell 4 — Extract project
import zipfile, os
with zipfile.ZipFile('QLSTM_with_GNN.zip', 'r') as z:
    z.extractall('/content/')
os.chdir('/content')
os.makedirs('checkpoints', exist_ok=True)
print('Files:', [f for f in sorted(os.listdir('.')) if not f.startswith('.')])
print('Data: ', sorted(os.listdir('data')))

In [ ]:
# Cell 4b — Mount Google Drive (checkpoint lưu vĩnh viễn)
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/QLSTM_checkpoints_p2'
os.makedirs(DRIVE_CKPT, exist_ok=True)

local_ckpt = '/content/checkpoints'
if os.path.exists(local_ckpt) and not os.path.islink(local_ckpt):
    for f in os.listdir(local_ckpt):
        shutil.move(os.path.join(local_ckpt, f), DRIVE_CKPT)
    os.rmdir(local_ckpt)
if not os.path.exists(local_ckpt):
    os.symlink(DRIVE_CKPT, local_ckpt)

print(f'checkpoints/ → {DRIVE_CKPT}')
existing = sorted(os.listdir('checkpoints'))
print(f'Existing checkpoints: {existing if existing else "(trống — train từ đầu)"}')

In [ ]:
# Cell 5 — Setup path + verify lightning.gpu
import sys, os
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/benchmark')

import pennylane as qml
import torch

try:
    qml.device('lightning.gpu', wires=4)
    BACKEND     = 'lightning.gpu'
    DIFF_METHOD = 'adjoint'
    DTYPE       = torch.float64
    print('QLSTM backend: lightning.gpu | adjoint | float64')
except Exception as e:
    print(f'lightning.gpu failed: {e}')
    BACKEND     = 'default.qubit'
    DIFF_METHOD = 'backprop'
    DTYPE       = torch.float32

# QAOA dùng lightning.qubit để tránh CUDA handle limit (500+ devices)
QAOA_SIM  = 'lightning.qubit'
QAOA_DIFF = 'adjoint'
print(f'QAOA  backend: {QAOA_SIM} | {QAOA_DIFF}')

In [ ]:
# Cell 6 — Model config (p=2, dim_h=16)
QAOA_P        = 2
DIM_H         = 16
N_QAOA_PARAMS = QAOA_P * 2           # 4
CH_GIN        = DIM_H * 8            # 128
CONCAT_SIZE   = N_QAOA_PARAMS + 1 + 4  # theta(4) + y(1) + h(4) = 9

print(f'QAOA p={QAOA_P} → n_qaoa_params={N_QAOA_PARAMS}')
print(f'GNN dim_h={DIM_H} → ch_gin={CH_GIN}')
print(f'concat_size={CONCAT_SIZE}')

In [ ]:
# Cell 7 — Shape test (verify trước khi train)
import networkx as nx, torch
from gnn import GNNEncoder, nx_to_pyg
from hypernet import HyperNet
from qlstm import QLSTMCell

gnn_t  = GNNEncoder(node_feature_dim=1, dim_h=DIM_H, n_qaoa_params=N_QAOA_PARAMS, qlstm_h_dim=4)
hyp_t  = HyperNet(embed_dim=CH_GIN, hidden_dim=128, n_vqcs=6, n_qlayers=2, n_qubits=4, concat_size=CONCAT_SIZE)
cell_t = QLSTMCell(n_qubits=4, n_qlayers=2, n_qaoa_params=N_QAOA_PARAMS, backend=BACKEND, diff_method=DIFF_METHOD)

G_t = nx.path_graph(6)
theta0, h0, e_G = gnn_t(nx_to_pyg(G_t))
out  = hyp_t(e_G)
W_in = out[0].squeeze(0).to(DTYPE)
b_in = out[1].squeeze(0).to(DTYPE)
phis = [p.squeeze(0).to(DTYPE) for p in out[2:]]
theta_t = theta0.squeeze(0).to(DTYPE)
h_prev  = h0.squeeze(0).to(DTYPE)
C_prev  = torch.zeros(4, dtype=DTYPE)
y_t     = torch.tensor(0.5, dtype=DTYPE)

with torch.no_grad():
    theta_next, h_t, C_t = cell_t(theta_t, y_t, h_prev, C_prev, W_in, b_in, *phis)

assert theta_next.shape == (N_QAOA_PARAMS,), f'theta_next: {theta_next.shape}'
assert e_G.shape        == (1, CH_GIN),      f'e_G: {e_G.shape}'
assert W_in.shape       == (4, CONCAT_SIZE), f'W_in: {W_in.shape}'
print('Shape test PASSED')
print(f'  e_G={tuple(e_G.shape)} | W_in={tuple(W_in.shape)} | theta_next={tuple(theta_next.shape)}')
print(f'  QLSTM: {BACKEND}/{DIFF_METHOD} | QAOA: {QAOA_SIM}/{QAOA_DIFF}')

In [ ]:
# Cell 8 — Training (auto-resume từ Drive)
import os, torch, random, time
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
import networkx as nx
from gnn import GNNEncoder, nx_to_pyg
from hypernet import HyperNet
from qlstm import QLSTMCell
from qaoa_maxcut import QAOAMaxCut
from train import load_or_create_dataset, run_pipeline, evaluate, CONFIG

cfg = dict(CONFIG)
cfg['qaoa_p'] = QAOA_P
cfg['dim_h']  = DIM_H

torch.manual_seed(cfg['seed'])
train_graphs, val_graphs = load_or_create_dataset(cfg)

# Khởi tạo models
gnn      = GNNEncoder(node_feature_dim=1, dim_h=DIM_H,
                       n_qaoa_params=N_QAOA_PARAMS, qlstm_h_dim=4)
hypernet = HyperNet(embed_dim=CH_GIN, hidden_dim=128, n_vqcs=6,
                     n_qlayers=2, n_qubits=4, concat_size=CONCAT_SIZE)
cell     = QLSTMCell(n_qubits=4, n_qlayers=2, n_qaoa_params=N_QAOA_PARAMS,
                      backend=BACKEND, diff_method=DIFF_METHOD)
optimizer = optim.Adam(
    list(gnn.parameters()) + list(hypernet.parameters()), lr=cfg['lr'])

# Auto-resume từ Drive
start_epoch = 1
best_val    = 0.0
latest_path = 'checkpoints/model_latest_p2.pth'
best_path   = 'checkpoints/model_best_p2.pth'

if os.path.exists(latest_path):
    ck = torch.load(latest_path, map_location='cpu', weights_only=False)
    gnn.load_state_dict(ck['gnn_state_dict'])
    hypernet.load_state_dict(ck['hypernet_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    start_epoch = ck['epoch'] + 1
    if os.path.exists(best_path):
        ck_best  = torch.load(best_path, map_location='cpu', weights_only=False)
        best_val = ck_best.get('val_ratio', 0.0)
    print(f'Resumed từ epoch {ck["epoch"]} | best_val={best_val:.4f}')
    print(f'Tiếp tục từ epoch {start_epoch} → {cfg["n_epochs"]}\n')
else:
    print('Không có checkpoint — train từ đầu\n')

n_gnn   = sum(p.numel() for p in gnn.parameters())
n_hyper = sum(p.numel() for p in hypernet.parameters())
print(f'Params — GNN: {n_gnn:,}  HyperNet: {n_hyper:,}')
print(f'Config — p={QAOA_P}, dim_h={DIM_H}, T={cfg["T"]}, epochs={cfg["n_epochs"]}')
print(f'QLSTM: {BACKEND}/{DIFF_METHOD} | QAOA: {QAOA_SIM}/{QAOA_DIFF} | dtype: {DTYPE}\n')

# Pre-create QAOA objects (lightning.qubit — tránh CUDA handle limit)
print('Pre-creating QAOA circuits...')
qaoa_train_cache = {id(G): QAOAMaxCut(G, p=QAOA_P, sim=QAOA_SIM, diff_method=QAOA_DIFF)
                    for G in train_graphs}
qaoa_val_cache   = {id(G): QAOAMaxCut(G, p=QAOA_P, sim=QAOA_SIM, diff_method=QAOA_DIFF)
                    for G in val_graphs}
print(f'Done: {len(qaoa_train_cache)} train + {len(qaoa_val_cache)} val circuits\n')

T = cfg['T']
P = cfg['qaoa_p']

for epoch in range(start_epoch, cfg['n_epochs'] + 1):
    gnn.train(); hypernet.train()
    loss_sum = 0.0
    t0 = time.time()
    random.shuffle(train_graphs)

    for G in train_graphs:
        optimizer.zero_grad()
        qaoa, thetas = run_pipeline(G, gnn, hypernet, cell, T, qaoa_p=P,
                                    qaoa_backend=QAOA_SIM, qaoa_diff_method=QAOA_DIFF,
                                    qaoa=qaoa_train_cache[id(G)])
        loss = sum(-qaoa.cost_from_theta(th) for th in thetas)
        loss.backward()
        clip_grad_norm_(list(gnn.parameters()) + list(hypernet.parameters()), 1.0)
        optimizer.step()
        loss_sum += loss.item()

    avg_loss = loss_sum / len(train_graphs)
    elapsed  = time.time() - t0

    # Lưu latest sau mỗi epoch → Drive (trước evaluate để không mất khi crash)
    torch.save({'epoch': epoch,
                'gnn_state_dict': gnn.state_dict(),
                'hypernet_state_dict': hypernet.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': cfg},
               latest_path)

    if epoch % cfg['log_every'] == 0:
        val_ratio = evaluate(gnn, hypernet, cell, val_graphs, T, qaoa_p=P,
                             qaoa_backend=QAOA_SIM, qaoa_diff_method=QAOA_DIFF,
                             qaoa_cache=qaoa_val_cache)
        tag = ''
        if val_ratio > best_val:
            best_val = val_ratio
            torch.save({'epoch': epoch,
                        'gnn_state_dict': gnn.state_dict(),
                        'hypernet_state_dict': hypernet.state_dict(),
                        'val_ratio': val_ratio, 'config': cfg},
                       best_path)
            tag = '  [saved best]'
        print(f'Epoch {epoch:3d}/{cfg["n_epochs"]} | loss={avg_loss:.4f} | '
              f'val={val_ratio:.4f} | best={best_val:.4f} | {elapsed:.1f}s{tag}', flush=True)
    else:
        print(f'Epoch {epoch:3d}/{cfg["n_epochs"]} | loss={avg_loss:.4f} | '
              f'{elapsed:.1f}s', flush=True)

print(f'\nTraining done. Best val_ratio={best_val:.4f}')
print('Checkpoints → Google Drive/QLSTM_checkpoints_p2/')

In [ ]:
# Cell 9 — Download checkpoints từ Drive về máy
import shutil
from google.colab import files
shutil.make_archive('checkpoints_p2', 'zip', 'checkpoints')
files.download('checkpoints_p2.zip')
print('Downloaded checkpoints_p2.zip')